# AIDM · Modelo de decisión: estado de la respuesta del discovery

Este cuaderno entrena un modelo de decisión para el **fallo 36**. En cada turno del discovery, el modelo lee el campo por el que se ha preguntado, la pregunta y la respuesta del cliente, y dice si la respuesta:

| Opción | Qué significa | Qué hace AIDM |
|---|---|---|
| `dato` | da el dato, aunque sea aproximado o negativo («no tenemos») | nada: el valor lo guarda el LLM |
| `no_lo_sabe` | dice que no lo sabe | guarda «Confirmado por el usuario: no lo sabe» |
| `no_aplica` | dice que no aplica a su proyecto | guarda «Confirmado por el usuario: no aplica» |
| `no_responde` | no contesta a lo preguntado | nada: el campo sigue pendiente |

El modelo no escribe texto: hace una sola pasada y devuelve una probabilidad por opción, así que se puede servir en CPU.

**Pasos:** instalar → datos → modelo base → entrenar → evaluar → probar en CPU con llama.cpp (como el Space) → subir el GGUF a Hugging Face.

**Antes de empezar:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
2. En la llave de la izquierda (*Secretos*), crea `HF_TOKEN` con un token de Hugging Face **de escritura** y dale acceso al cuaderno.
3. Rellena tu usuario de Hugging Face en la celda de configuración.

Los ejemplos son inventados (`modelo_decision/generar_datos.py`): no hay datos de clientes.

## 1. Instalar Unsloth
Es la celda del cuaderno oficial de Unsloth. Tarda unos minutos.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 2. Configuración

In [ ]:
USUARIO_HF = "tu_usuario"            # tu usuario de Hugging Face
REPO_MODELO = "aidm-decision"        # repo privado donde se sube el GGUF
MODELO_BASE = "unsloth/Qwen3.5-0.8B"
RAMA = "https://raw.githubusercontent.com/ruBBBBBBa/AIDM/pruebas-rubengs"  # de aquí se bajan los datos
UMBRAL = 0.7                         # la misma confianza mínima que decisiones.CONFIANZA_MINIMA en AIDM

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert USUARIO_HF != "tu_usuario", "Pon tu usuario de Hugging Face en USUARIO_HF"

## 3. Datos
Se bajan de la rama. Si todavía no los has subido, súbelos a mano a la carpeta `datos/` desde el panel de archivos (los dos `.jsonl` de `modelo_decision/datos/`).

In [ ]:
import json, os, urllib.request
from collections import Counter

os.makedirs("datos", exist_ok=True)
for nombre in ("entrenamiento", "prueba"):
    ruta = f"datos/{nombre}.jsonl"
    if not os.path.exists(ruta):
        try:
            urllib.request.urlretrieve(f"{RAMA}/modelo_decision/datos/{nombre}.jsonl", ruta)
        except Exception as exc:
            print(f"No se pudo bajar {nombre}.jsonl ({exc}): súbelo a la carpeta datos/")

def leer(ruta):
    with open(ruta, encoding="utf-8") as f:
        return [json.loads(linea) for linea in f if linea.strip()]

entrenamiento, prueba = leer("datos/entrenamiento.jsonl"), leer("datos/prueba.jsonl")
for nombre, filas in (("entrenamiento", entrenamiento), ("prueba", prueba)):
    print(nombre, len(filas), dict(Counter(f["gold"]["estado"] for f in filas)))
print()
print(entrenamiento[0]["state"])
print("→", entrenamiento[0]["gold"]["estado"])

## 4. Modelo base
Qwen3.5-0.8B en 4 bits con LoRA (r=16), como recomienda la guía de Unsloth. La cabeza de decisión es nueva y siempre se entrena.

In [ ]:
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = MODELO_BASE,
    max_seq_length = 1024,   # los ejemplos ocupan unos 500 tokens con el esquema
    load_in_4bit = True,
)
model = FastDecisionModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
items, informe = FastDecisionModel.build_dataset(entrenamiento, tokenizer, model)
print(f"Entrenamiento: descartados {informe['skipped']} de {informe['total']}", informe["reason"] or "")
train_items, eval_items = FastDecisionModel.split_holdout(items, seed = 3407)  # para calibrar
test_items, informe_prueba = FastDecisionModel.build_dataset(prueba, tokenizer, model)
print(f"{len(train_items)} para entrenar, {len(eval_items)} apartados para calibrar, {len(test_items)} de prueba")

Antes de entrenar, la cabeza es nueva y acierta más o menos por azar:

In [ ]:
FastDecisionModel.evaluate(model, tokenizer, test_items)

## 5. Entrenar
Dos épocas, como en la guía. En una T4, Qwen3.5 entrena en float32 (la T4 no tiene bfloat16), así que va lento: el cuaderno oficial tarda unas 2 h con el modelo de 4B; con 0.8B debería tardar bastante menos.

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_items,
    eval_dataset = eval_items,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        logging_steps = 10,
        output_dir = "outputs",
        report_to = "none",
        seed = 3407,
    ),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} min de entrenamiento")

**Opcional:** guarda los adaptadores en un repo privado. Si Colab corta la sesión, puedes cargarlos y exportar sin volver a entrenar (`FastDecisionModel.from_pretrained(f"{USUARIO_HF}/{REPO_MODELO}-lora", load_in_4bit=True)`).

In [ ]:
model.push_to_hub(f"{USUARIO_HF}/{REPO_MODELO}-lora", tokenizer, token = HF_TOKEN, private = True)

## 6. Calibrar y evaluar
La calibración ajusta las probabilidades para que una confianza de 0,9 acierte más o menos 9 de cada 10 veces. Luego se mide con la prueba, que usa frases que el modelo no ha visto.

In [ ]:
print(FastDecisionModel.calibrate(model, tokenizer, eval_items))
print(FastDecisionModel.evaluate(model, tokenizer, test_items))

In [ ]:
import pandas as pd

FastDecisionModel.for_inference(model)
filas = []
for f in prueba:
    r = FastDecisionModel.predict(model, tokenizer, f["state"], f["questions"])["estado"]
    filas.append({"real": f["gold"]["estado"], "modelo": r["answer"], "confianza": r.get("confidence"),
                  "idioma": f["idioma"], "campo": f["campo"], "respuesta": f["state"].split("Respuesta del cliente: ")[-1]})
tabla = pd.DataFrame(filas)
tabla["acierta"] = tabla.real == tabla.modelo

print(f"Aciertos: {tabla.acierta.mean():.1%}")
print(tabla.groupby("idioma").acierta.mean().map("{:.1%}".format).to_string())
print()
print(pd.crosstab(tabla.real, tabla.modelo))

# Lo que importa en AIDM: cuando dice no_lo_sabe o no_aplica con confianza suficiente, se guarda sin el LLM.
cerradas = tabla[tabla.modelo.isin(["no_lo_sabe", "no_aplica"]) & (tabla.confianza >= UMBRAL)]
reales = tabla[tabla.real.isin(["no_lo_sabe", "no_aplica"])]
print(f"\nCon confianza ≥ {UMBRAL}: marca {len(cerradas)} como no_lo_sabe o no_aplica y acierta {cerradas.acierta.mean():.1%}; "
      f"recoge {len(cerradas[cerradas.acierta])} de los {len(reales)} que hay.")
tabla[~tabla.acierta]

## 7. Probar a mano
El texto del estado es el mismo que arma `decisiones.texto_estado()` en AIDM.

In [ ]:
PREGUNTA = entrenamiento[0]["questions"]["estado"]
DESCRIPCIONES = {}
for f in entrenamiento:
    linea = f["state"].split("\n")[0]
    DESCRIPCIONES[f["campo"]] = linea[linea.index(" (") + 2:-1]

def estado(campo, pregunta, respuesta):
    return (f"Dato que se pregunta: {campo} ({DESCRIPCIONES[campo]})\n"
            f"Pregunta del asistente: {pregunta}\n"
            f"Respuesta del cliente: {respuesta}")

def probar(campo, pregunta, respuesta):
    r = FastDecisionModel.predict(model, tokenizer, estado(campo, pregunta, respuesta), {"estado": PREGUNTA})["estado"]
    print(f"{r['answer']:<12} {r.get('confidence', 0):.2f}  {campo}: «{respuesta}»")

probar("tipo_de_arquitectura", "¿Tenéis preferencia de arquitectura: data warehouse, data lake o lakehouse?", "No lo sé")
probar("tiene_equipo_datos", "¿Tenéis equipo de datos propio?", "No, no hay equipo de datos")
probar("regulacion_y_compliance", "¿Hay alguna normativa que debamos tener en cuenta?", "Eso no aplica, no tenemos datos personales")
probar("volumen_de_datos", "¿Qué volumen de datos manejáis?", "Somos una empresa de transporte con 300 camiones")
probar("herramienta_de_visualizacion", "¿Usáis alguna herramienta de BI?", "No estoy seguro, creo que en finanzas usan Power BI")
probar("plazo_objetivo", "When would you need it up and running?", "No idea, management decides that")

## 8. Exportar a GGUF y probarlo en CPU
El Space sirve el modelo con llama.cpp en CPU (2 vCPU), así que aquí se prueba igual: se baja `llama-server` de la última versión de llama.cpp, se arranca con 2 hilos y se le hacen las mismas peticiones que hará AIDM a `/v1/systemone`.

In [ ]:
model.save_pretrained_gguf("exportado", tokenizer, quantization_method = "q8_0", token = HF_TOKEN)
GGUF = "exportado/gguf/model-Q8_0.gguf"
!ls -lh exportado/gguf

In [ ]:
import glob, statistics, subprocess, tarfile, time, zipfile
import requests

version = json.load(urllib.request.urlopen("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest"))
archivo = next(a for a in version["assets"] if a["name"].endswith(("bin-ubuntu-x64.tar.gz", "bin-ubuntu-x64.zip")))
print("llama.cpp", version["tag_name"], "·", archivo["name"])
urllib.request.urlretrieve(archivo["browser_download_url"], archivo["name"])
if archivo["name"].endswith(".zip"):
    zipfile.ZipFile(archivo["name"]).extractall("llama_bin")
else:
    tarfile.open(archivo["name"]).extractall("llama_bin")
servidor = glob.glob("llama_bin/**/llama-server", recursive = True)[0]
os.chmod(servidor, 0o755)

proceso = subprocess.Popen(
    [servidor, "-m", GGUF, "--port", "8080", "-c", "2048", "-b", "2048", "-ub", "2048", "-t", "2", "--parallel", "1"],
    env = {**os.environ, "LD_LIBRARY_PATH": os.path.dirname(servidor), "CUDA_VISIBLE_DEVICES": ""},
    stdout = open("llama_server.log", "w"), stderr = subprocess.STDOUT,
)
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout = 2).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("llama-server no arranca: mira llama_server.log")
print("llama-server listo")

In [ ]:
def decidir(texto_estado):
    r = requests.post("http://127.0.0.1:8080/v1/systemone",
                      json = {"state": texto_estado, "questions": {"estado": PREGUNTA}}, timeout = 120)
    r.raise_for_status()
    return r.json()["answers"]["estado"]

tiempos, aciertos = [], 0
for f in prueba:
    inicio = time.time()
    d = decidir(f["state"])
    tiempos.append(time.time() - inicio)
    aciertos += d["choice"] == f["gold"]["estado"]
print(f"llama-server en CPU (2 hilos): acierta {aciertos} de {len(prueba)} ({aciertos / len(prueba):.1%})")
print(f"Tiempo por petición: mediana {statistics.median(tiempos):.2f} s, máximo {max(tiempos):.2f} s")
print("Respuesta de ejemplo:", decidir(prueba[0]["state"]))

Tiene que acertar más o menos lo mismo que en el paso 6. Si acierta mucho menos, la conversión a GGUF ha fallado y no merece la pena montar el Space.

## 9. Subir el GGUF a Hugging Face (repo privado)

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token = HF_TOKEN)
repo = api.create_repo(f"{USUARIO_HF}/{REPO_MODELO}", private = True, exist_ok = True).repo_id
api.upload_folder(folder_path = "exportado/gguf", repo_id = repo)
print(f"Subido: https://huggingface.co/{repo}")
proceso.terminate()

## 10. Siguiente: el Space
1. En Hugging Face, *New Space*: SDK **Docker**, hardware **CPU basic** (gratis) y **privado**. Llámalo `aidm-decision`.
2. Sube los dos archivos de `modelo_decision/space/` (`Dockerfile` y `README.md`). En el `Dockerfile`, cambia `tu_usuario` por el tuyo.
3. En *Settings → Variables and secrets*, añade el secreto `HF_TOKEN` (de lectura) para que el Space pueda bajar el modelo privado.
4. Cuando esté *Running*, en el `.env` de AIDM:
```
AIDM_DECISION_URL=https://tu_usuario-aidm-decision.hf.space
HF_TOKEN=hf_...
```